# Coleta de Dados: Crawlers e Web Scraping

**Notebook único — os 3 projetos em um só arquivo**

Este notebook reúne, em sequência, os três projetos da aula:

1. **Projeto 1 — `requests` + `BeautifulSoup`** (site de clima)
2. **Projeto 2 — `BeautifulSoup`** em site real (`books.toscrape.com`)
3. **Projeto 3 — Comparando o dólar turismo** (APIs JSON + `pandas.read_html`)

> Sugestão de uso no Colab: execute as células **de cima para baixo**. Cada
> projeto começa com uma divisória e tem sua própria célula de instalação, então
> também é possível rodar apenas o trecho que você quiser.

---


# Projeto 1 — Coleta de dados com `requests` e `BeautifulSoup`

**Coleta de Dados: Crawlers e Web Scraping**

Neste primeiro notebook damos os dois primeiros passos do web scraping:

1. **Baixar** o HTML de uma página com a biblioteca `requests`.
2. **Extrair** os dados desse HTML de forma organizada com o `BeautifulSoup`.

O site usado como exemplo é um painel de clima (dados já presentes no HTML):

> https://guilhermeonrails.github.io/clima-tempo/

> **Conceitos-chave**
> - *Coleta de dados*: recolher informações da web de forma automática.
> - *Crawler*: programa que navega entre páginas seguindo links.
> - *Web scraping*: extrair dados específicos de dentro de uma página.


## Preparando o ambiente

No Google Colab instalamos as bibliotecas com `pip`. O `requests` faz as
requisições HTTP; o `beautifulsoup4` (com o motor `lxml`) organiza o HTML.


In [ ]:
!pip install -q requests beautifulsoup4 lxml
print('Instalação feita com sucesso ✅')

Instalação feita com sucesso ✅


## Parte 1 — Baixando o HTML com `requests`

Quando você abre um site, o navegador faz uma **requisição** ao servidor e
recebe de volta o HTML da página. A biblioteca `requests` faz exatamente isso
pelo código. O método `get` significa "me entregue essa página".

Vamos observar três coisas:
- `res.status_code` → código de status (**200** = OK);
- `len(res.text)` → quantidade de caracteres do HTML;
- `res.text` → o HTML **cru** (texto puro).


In [ ]:
import requests

URL = "https://guilhermeonrails.github.io/clima-tempo/"

# Faz a requisição GET (pede o conteúdo da página)
res = requests.get(URL)

print("status_code:", res.status_code)          # 200 = deu certo
print("Caracteres no res.text:", len(res.text))  # tamanho do HTML

# Mostra apenas os primeiros 1000 caracteres do HTML cru
print("\n----- HTML CRU (início) -----")
print(res.text[:1000])

status_code: 200
Caracteres no res.text: 4598

----- HTML CRU (início) -----
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Painel de Clima</title>
  <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gradient-to-br from-blue-200 to-white min-h-screen p-6 font-sans">

  <h1 class="text-4xl font-bold text-center text-blue-800 mb-10">🌎 Painel de Clima - Capitais do Mundo</h1>

  <div class="grid grid-cols-1 sm:grid-cols-2 md:grid-cols-3 gap-6 max-w-6xl mx-auto">

    <div class="bg-white p-6 rounded-2xl shadow-xl border border-blue-100">
      <h2 class="text-2xl font-semibold text-blue-700 mb-2">São Paulo</h2>
      <p class="text-gray-600">Temperatura: <span id="temperatura" class="font-bold">26°C</span></p>
      <p class="text-gray-600">Clima: <span id="clima" class="font-bold">Parcialmente Nublado</span></p>
      <p class="text-gray-600">Umidade: <span 

### O problema do HTML cru

Repare que `res.text` traz **tudo junto**: tags, classes e conteúdo misturados.
Para a biblioteca `requests`, o HTML é só um texto gigante — ela não entende a
estrutura. Se quiséssemos só as cidades e temperaturas, teríamos muito trabalho.

É aí que entra o **BeautifulSoup**.


## Parte 2 — Extraindo os dados com `BeautifulSoup`

O `BeautifulSoup` transforma o HTML numa **árvore navegável**. A partir dela,
pedimos exatamente o que queremos usando:

- `soup.find(tag)` → o **primeiro** elemento;
- `soup.find_all(tag, class_=...)` → **todos** os elementos;
- `elemento.get_text()` → apenas o **texto**, sem as tags.

No painel de clima, cada cidade é um cartão `div` com a classe `bg-white`.
Dentro dele: a cidade fica num `<h2>` e os três valores (temperatura, clima e
umidade) ficam em `<span>` com a classe `font-bold`, **nessa ordem**.

> Curiosidade: a página repete o mesmo `id` em vários `span` (algo inválido em
> HTML). Por isso **não** buscamos por `id` — buscamos por **classe** dentro de
> cada cartão, o que é mais confiável.


In [ ]:
from bs4 import BeautifulSoup

# Entrega o HTML (baixado com requests) ao BeautifulSoup
soup = BeautifulSoup(res.text, "lxml")

# find: primeiro elemento. Aqui, o título principal da página (<h1>)
titulo = soup.find("h1")
print("Título da página:", titulo.get_text(strip=True))

Título da página: 🌎 Painel de Clima - Capitais do Mundo
Cidades encontradas: 9


In [ ]:
# find_all: todos os cartões de cidade (div com a classe bg-white)
cartoes = soup.find_all("div", class_="bg-white")
print("Cidades encontradas:", len(cartoes))

Cidades encontradas: 9


In [ ]:
# Para cada cartão, extraímos cidade + os três valores (na ordem dos spans)
dados = []
for cartao in cartoes:
    cidade = cartao.find("h2").get_text(strip=True)
    valores = cartao.find_all("span", class_="font-bold")  # [temp, clima, umidade]
    temperatura = valores[0].get_text(strip=True)
    clima       = valores[1].get_text(strip=True)
    umidade     = valores[2].get_text(strip=True)

    dados.append({
        "cidade": cidade,
        "temperatura": temperatura,
        "clima": clima,
        "umidade": umidade,
    })

    print(f"{cidade:15} | {temperatura:6} | {clima:22} | umidade {umidade}")

São Paulo       | 26°C   | Parcialmente Nublado   | umidade 55%
Brasília        | 29°C   | Ensolarado             | umidade 40%
Rio de Janeiro  | 31°C   | Muito Quente           | umidade 60%
Buenos Aires    | 22°C   | Chuva Leve             | umidade 70%
Londres         | 15°C   | Nublado                | umidade 80%
Paris           | 17°C   | Chuvisco               | umidade 75%
Tóquio          | 24°C   | Ensolarado             | umidade 65%
Nova York       | 20°C   | Trovoadas              | umidade 67%
Cairo           | 35°C   | Seco                   | umidade 20%


### Bônus: organizando em uma tabela

Como já temos uma lista de dicionários, podemos jogar tudo num `DataFrame` do
`pandas` (já disponível no Colab) e ter uma tabela pronta para análise ou para
salvar em CSV.


In [ ]:
import pandas as pd

df = pd.DataFrame(dados)
df

,cidade,temperatura,clima,umidade
0,São Paulo,26°C,Parcialmente Nublado,55%
1,Brasília,29°C,Ensolarado,40%
2,Rio de Janeiro,31°C,Muito Quente,60%
3,Buenos Aires,22°C,Chuva Leve,70%
4,Londres,15°C,Nublado,80%
5,Paris,17°C,Chuvisco,75%
6,Tóquio,24°C,Ensolarado,65%
7,Nova York,20°C,Trovoadas,67%
8,Cairo,35°C,Seco,20%


In [ ]:
# Salvando em CSV (aparece no painel de arquivos do Colab)
df.to_csv("clima.csv", index=False)
print("Arquivo clima.csv salvo!")

Arquivo clima.csv salvo!


## Conclusão

- O `requests` **baixa** o HTML, mas entrega um texto cru difícil de tratar.
- O `BeautifulSoup` **organiza** esse HTML e permite extrair dados com clareza
  usando `find`, `find_all` e `get_text`.
- Juntos, eles transformam uma página em **dados estruturados**.

**Ética:** antes de raspar um site real, verifique o `robots.txt` e os termos
de uso, e nunca sobrecarregue o servidor com requisições em excesso.

No **Projeto 2** aprofundamos o BeautifulSoup em um site de verdade
(`books.toscrape.com`).



---


# Projeto 2 — Web scraping com `BeautifulSoup`

**Coleta de Dados: Crawlers e Web Scraping**

Agora aplicamos o `BeautifulSoup` em um **site real**, criado especificamente
para praticar scraping:

> https://books.toscrape.com

É uma livraria fictícia com centenas de livros, preço, avaliação e disponibilidade.
Vamos extrair esses dados e ainda percorrer **várias páginas** (paginação).


## Preparando o ambiente

In [ ]:
!pip install -q requests beautifulsoup4 lxml
print('Instalação feita com sucesso ✅')

Instalação feita com sucesso ✅


## Baixando a primeira página

Como no Projeto 1, usamos o `requests` para baixar o HTML e conferimos o
`status_code`.


In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://books.toscrape.com/"
res = requests.get(URL)
print("status_code:", res.status_code)

soup = BeautifulSoup(res.text)

status_code: 200


## Entendendo a estrutura

Cada livro está dentro de um `<article class="product_pod">`. Dentro dele:

- **título** → fica no atributo `title` do link `<a>` (dentro de um `<h3>`);
- **preço** → `<p class="price_color">`;
- **avaliação** → `<p class="star-rating Three">` (a nota é a *segunda* classe);
- **disponibilidade** → `<p class="instock availability">`.

> Note uma novidade: às vezes o dado está em um **atributo** (como `title`), e
> não no texto. Acessamos atributos como em um dicionário: `tag["title"]`.


In [ ]:
livros = soup.find_all("article", class_="product_pod")
print("Livros nesta página:", len(livros))

dados = []
for livro in livros:
    titulo = livro.find("h3").find("a")["title"]          # atributo title
    preco  = livro.find("p", class_="price_color").get_text(strip=True)
    # A classe star-rating tem 2 valores, ex.: ["star-rating", "Three"]
    nota   = livro.find("p", class_="star-rating")["class"][1]
    estoque = livro.find("p", class_="instock").get_text(strip=True)

    dados.append({"titulo": titulo, "preco": preco, "nota": nota, "estoque": estoque})
    print(f"{nota:6} | {preco:8} | {titulo}")

Livros nesta página: 20
Three  | Â£51.77  | A Light in the Attic
One    | Â£53.74  | Tipping the Velvet
One    | Â£50.10  | Soumission
Four   | Â£47.82  | Sharp Objects
Five   | Â£54.23  | Sapiens: A Brief History of Humankind
One    | Â£22.65  | The Requiem Red
Four   | Â£33.34  | The Dirty Little Secrets of Getting Your Dream Job
Three  | Â£17.93  | The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Four   | Â£22.60  | The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
One    | Â£52.15  | The Black Maria
Two    | Â£13.99  | Starving Hearts (Triangular Trade Trilogy, #1)
Four   | Â£20.66  | Shakespeare's Sonnets
Five   | Â£17.46  | Set Me Free
Five   | Â£52.29  | Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
Five   | Â£35.02  | Rip it Up and Start Again
Three  | Â£57.25  | Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
One    | Â£23.88  | Olio
One    | Â£37.59

## Outra forma de buscar: `select` (seletores CSS)

Além de `find`/`find_all`, o BeautifulSoup aceita **seletores CSS** com `select`.
Por exemplo, `article.product_pod h3 a` pega os links de título de todos os livros.


In [ ]:
# Apenas os títulos, usando seletor CSS
for livro in soup.select("article.product_pod h3 a"):
    print("-", livro["title"])

- A Light in the Attic
- Tipping the Velvet
- Soumission
- Sharp Objects
- Sapiens: A Brief History of Humankind
- The Requiem Red
- The Dirty Little Secrets of Getting Your Dream Job
- The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
- The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
- The Black Maria
- Starving Hearts (Triangular Trade Trilogy, #1)
- Shakespeare's Sonnets
- Set Me Free
- Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
- Rip it Up and Start Again
- Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
- Olio
- Mesaerion: The Best Science Fiction Stories 1800-1849
- Libertarianism for Beginners
- It's Only the Himalayas


## Percorrendo várias páginas (paginação)

Um **crawler** simples: o catálogo tem várias páginas no padrão
`catalogue/page-N.html`. Vamos percorrer as 3 primeiras e juntar tudo.

> Boas práticas: usamos `time.sleep` entre as requisições para não sobrecarregar
> o servidor.


In [ ]:
import time

todos = []
for pagina in range(1, 4):  # páginas 1, 2 e 3
    url = f"https://books.toscrape.com/catalogue/page-{pagina}.html"
    r = requests.get(url)
    if r.status_code != 200:
        break
    s = BeautifulSoup(r.text, "lxml")
    for livro in s.find_all("article", class_="product_pod"):
        todos.append({
            "titulo": livro.find("h3").find("a")["title"],
            "preco": livro.find("p", class_="price_color").get_text(strip=True),
            "nota": livro.find("p", class_="star-rating")["class"][1],
        })
    print(f"Página {pagina}: total acumulado = {len(todos)} livros")
    time.sleep(1)  # pausa educada entre as requisições

Página 1: total acumulado = 20 livros
Página 2: total acumulado = 40 livros
Página 3: total acumulado = 60 livros


## Organizando em tabela e salvando

In [ ]:
import pandas as pd

df = pd.DataFrame(todos)
df.to_csv("livros.csv", index=False)
print("Total de livros coletados:", len(df))
df.head(10)

Total de livros coletados: 60


,titulo,preco,nota
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One


## Conclusão

- Vimos como extrair dados que estão no **texto** e em **atributos** (`title`).
- Conhecemos o `select` (seletores CSS) como alternativa ao `find_all`.
- Demos um passo de **crawler**, percorrendo várias páginas.

**Ética:** o `books.toscrape.com` é feito para praticar. Em sites reais, sempre
verifique o `robots.txt` e os termos de uso.

No **Projeto 3** veremos sites que **exigem interação** (login e cliques) — onde
o `requests` não basta e entra o **Selenium**.



---


# Projeto 3 — Comparando o Dólar Turismo de várias fontes

**Coleta de Dados: Crawlers e Web Scraping**

Neste projeto saímos do exemplo didático e vamos a um **problema real**:
descobrir **onde o dólar turismo está mais barato**. Para isso, coletamos a
cotação de **várias fontes** e comparamos.

Vamos usar **duas abordagens** de coleta, ambas simples e que rodam direto no
Colab (instalação via `pip`):

1. **Consumir APIs (JSON) com `requests`** — muitos serviços entregam os dados
   já prontos em JSON. É mais simples e estável do que raspar HTML.
2. **`pandas.read_html`** — lê **tabelas HTML** de uma página com **uma linha**
   de código.
3. Reaproveitamos o **`BeautifulSoup`** (Projetos 1 e 2) para um site cujo valor
   está dentro de um campo `<input>`.

> **Por que NÃO usamos Selenium aqui?** Selenium controla um navegador inteiro —
> é pesado e só se justifica quando há **interação** (login, cliques) ou
> conteúdo criado por **JavaScript**. Para cotações, os dados estão em APIs e em
> HTML simples, então ferramentas mais leves resolvem melhor. **Escolher a
> ferramenta certa para cada caso é uma habilidade central do profissional de
> dados.**

### Dólar comercial x dólar turismo
- **Comercial:** taxa usada em grandes operações financeiras (referência).
- **Turismo:** o que você paga ao comprar moeda em espécie para viajar — costuma
  ser **mais caro** (tem um *spread*).

### Nossas fontes
| Fonte | O que é | Como coletamos |
|---|---|---|
| **Wise** | Provedor real de câmbio | API JSON (`requests`) |
| **DolarHoje** | Site de cotação turismo | HTML + `BeautifulSoup` |
| **x-rates** | Site com tabela de moedas | Tabela HTML + `pandas.read_html` |

> ⚠️ **Atenção (sites reais):** ao contrário dos projetos anteriores, estes são
> sites/serviços de verdade. Eles **podem mudar de layout, sair do ar ou
> bloquear** o acesso a qualquer momento — então o código pode precisar de
> ajustes no futuro. Por isso protegemos cada coleta com `try/except`.


## Preparando o ambiente

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas
print('Instalação feita com sucesso ✅')

Instalação feita com sucesso ✅


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

# Cabeçalho de navegador: alguns sites respondem melhor quando nos identificamos
HEAD = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Aqui vamos juntando o resultado de cada fonte
resultados = []

def para_float(texto):
    """Converte um preço em texto BR (ex.: '5,38') para float (5.38)."""
    return float(str(texto).replace(".", "").replace(",", "."))

## Fonte 1 — Wise (provedor real) via API

A **Wise** é um provedor de câmbio internacional. Ela expõe um endereço público
que devolve a sua taxa em JSON. Note que é o **mesmo `requests`** — só muda a URL
e o formato do retorno.

> A taxa da Wise é próxima da **comercial** (taxa de mercado); as tarifas da Wise
> são cobradas à parte. Guardamos como referência "comercial".


In [ ]:
try:
    url = "https://wise.com/rates/live?source=USD&target=BRL"
    valor = requests.get(url, headers=HEAD, timeout=15).json()["value"]
    valor = float(valor)

    print(f"Wise - taxa USD->BRL: R$ {valor:.4f}")
    resultados.append({"fonte": "Wise", "tipo": "comercial", "preco_venda_usd": valor})
except Exception as e:
    print("Falha ao coletar da Wise:", e)

Wise - taxa USD->BRL: R$ 5.1731


## Fonte 2 — Site real com `BeautifulSoup` (DolarHoje)

Nem todo dado está numa API. No site **DolarHoje**, a cotação do dólar turismo
fica dentro de um campo de formulário: `<input id="nacional" value="5,38">`.

Aqui reaproveitamos o `BeautifulSoup` (Projetos 1 e 2): baixamos o HTML com
`requests`, localizamos o `input` pelo `id` e lemos o **atributo** `value`.


In [ ]:
try:
    url = "https://dolarhoje.com/dolar-turismo/"
    html = requests.get(url, headers=HEAD, timeout=15).text
    soup = BeautifulSoup(html, "lxml")

    campo = soup.find("input", id="nacional")   # campo com a cotação
    valor = para_float(campo["value"])           # ex.: "5,38" -> 5.38

    print(f"DolarHoje - turismo: R$ {valor:.4f}")
    resultados.append({"fonte": "DolarHoje", "tipo": "turismo", "preco_venda_usd": valor})
except Exception as e:
    print("Falha ao coletar do DolarHoje:", e)

DolarHoje - turismo: R$ 5.3800


## Fonte 3 — Tabela HTML com `pandas.read_html` (x-rates)

Quando os dados estão em uma **tabela HTML** (`<table>`), o `pandas` resolve em
uma linha: `pd.read_html(...)` lê **todas** as tabelas da página e devolve uma
lista de `DataFrame`. É a forma mais rápida de raspar dados tabulares.

No site **x-rates**, procuramos a linha do **Brazilian Real** para achar quanto
vale 1 dólar (comercial).


In [ ]:
try:
    url = "https://www.x-rates.com/table/?from=USD&amount=1"
    html = requests.get(url, headers=HEAD, timeout=15).text

    tabelas = pd.read_html(StringIO(html))  # lista de DataFrames
    print("Tabelas encontradas na página:", len(tabelas))

    valor = None
    for t in tabelas:
        txt = t.astype(str)
        mask = txt.apply(lambda c: c.str.contains("Brazil", case=False, na=False)).any(axis=1)
        if mask.any():
            linha = t[mask].iloc[0].tolist()
            # pega o primeiro número plausível (1 USD ~ alguns reais)
            for x in linha:
                try:
                    n = float(x)
                    if 1 < n < 100:
                        valor = n
                        break
                except (ValueError, TypeError):
                    pass
            break

    if valor:
        print(f"x-rates - USD->BRL (comercial): R$ {valor:.4f}")
        resultados.append({"fonte": "x-rates", "tipo": "comercial", "preco_venda_usd": valor})
    else:
        print("Não encontrei a linha do Real na tabela.")
except Exception as e:
    print("Falha ao coletar do x-rates:", e)

Tabelas encontradas na página: 2
x-rates - USD->BRL (comercial): R$ 5.1757


## Comparando tudo e achando o melhor câmbio

Agora juntamos todas as fontes em uma tabela (`DataFrame`), ordenamos pelo
**preço de venda** e descobrimos onde o dólar está mais barato.

Para uma comparação **justa**, separamos por tipo: o turista deve olhar a linha
**turismo** (o que ele realmente paga). O **menor** valor é o melhor câmbio.


In [ ]:
df = pd.DataFrame(resultados).sort_values("preco_venda_usd").reset_index(drop=True)
print("Cotações coletadas (R$ por 1 US$), da mais barata para a mais cara:\n")
print(df.to_string(index=False))

df.to_csv("cotacoes_dolar.csv", index=False)

# Melhor câmbio considerando o DÓLAR TURISMO (o que o turista paga)
turismo = df[df["tipo"] == "turismo"]
if not turismo.empty:
    melhor = turismo.iloc[0]
    print(f"\n>>> Dólar TURISMO: {melhor['fonte']} a R$ {melhor['preco_venda_usd']:.4f}")

Cotações coletadas (R$ por 1 US$), da mais barata para a mais cara:

    fonte      tipo  preco_venda_usd
     Wise comercial         5.173150
  x-rates comercial         5.175717
DolarHoje   turismo         5.380000

>>> Dólar TURISMO: DolarHoje a R$ 5.3800


## Conclusão e cuidados

- Aprendemos que **coletar dados nem sempre é raspar HTML**: muitas vezes a
  forma mais simples e estável é **consumir uma API (JSON)** com `requests`.
- Vimos o `pandas.read_html`, que raspa **tabelas** em uma linha.
- Reaproveitamos o `BeautifulSoup` para ler um valor dentro de um `<input>`.
- Consolidamos tudo em um `DataFrame` e respondemos a uma pergunta real:
  **onde o dólar turismo está mais barato?**

> **Comparação justa:** o dólar **comercial** (Wise/x-rates) é só **referência** —
> não é o que o turista paga. Além do *spread* do turismo, há **IOF** e
> **tarifas** que variam por instituição. Para uma decisão real, inclua esses
> custos. Aqui o objetivo é **didático**: mostrar a coleta e a comparação.

**Ética e responsabilidade:** estes são serviços reais. Consulte sempre o
`robots.txt` e os **Termos de Uso** (vários proíbem scraping), não faça
requisições em excesso e, quando existir **API oficial**, prefira-a — foi
exatamente o que fizemos com a AwesomeAPI e a Wise.



---

**MBA USP ESALQ - Data Science**
